# 2.BUILDING THE SILVER LAYER : CLEANING AND VALIDATING DATA
## 2.1 Reading Data from the bronze table

In [0]:
catalog       = "hr_catalogue_business_case"
bronze_schema = "hr_raw" 

bronze_contracts     = spark.table(f"{catalog}.{bronze_schema}.contracts")
bronze_workplan      = spark.table(f"{catalog}.{bronze_schema}.workplan")
bronze_salary        = spark.table(f"{catalog}.{bronze_schema}.salary")
bronze_absences      = spark.table(f"{catalog}.{bronze_schema}.absences")
bronze_absence_types = spark.table(f"{catalog}.{bronze_schema}.absence_types")
bronze_postcodes     = spark.table(f"{catalog}.{bronze_schema}.postcodes")

bronze_tables = {
    "contracts":     bronze_contracts,
    "workplan":      bronze_workplan,
    "salary":        bronze_salary,
    "absences":      bronze_absences,
    "absence_types": bronze_absence_types,
    "postcodes":     bronze_postcodes,
}

for name, df in bronze_tables.items():
    print(f"Total Bronze records - {name:<15}: {df.count()}")

In [0]:
display(bronze_contracts)

## 2.2 : First cleaning step : Removing duplicates

In [0]:
def dropduplicates(df):
    return df.dropDuplicates()

df_deduplicated = {}

for name, df in bronze_tables.items():
    before = df.count()
    df_clean = dropduplicates(df)
    after = df_clean.count()
    df_deduplicated[name] = df_clean
    print(f"{name:<15} {before:>10} {after:>10} {before - after:>10}")

In [0]:
df_contracts = df_deduplicated["contracts"]
df_workplan  = df_deduplicated["workplan"]
df_salary    = df_deduplicated["salary"]
df_absences  = df_deduplicated["absences"]
df_abs_types = df_deduplicated["absence_types"]
df_postcodes = df_deduplicated["postcodes"]

## 2.3. Verify if needed to standardise

In [0]:
from pyspark.sql.functions import col, countDistinct, lower, trim

tables = {
    "contracts":     df_contracts,
    "workplan":      df_workplan,
    "salary":        df_salary,
    "absences":      df_absences,
    "absence_types": df_abs_types,
    "postcodes":     df_postcodes,
}
for name, df in tables.items():
    cols = [c for c in df.columns if not c.startswith("_")]
    exprs = []
    for c in cols:
        exprs.append(countDistinct(col(c)).alias(f"{c}__raw"))
        exprs.append(countDistinct(lower(trim(col(c)))).alias(f"{c}__norm"))
    result = df.agg(*exprs).collect()[0]

    for c in cols:
        raw, norm = result[f"{c}__raw"], result[f"{c}__norm"]
        if raw != norm:
            print(f"⚠️  {name}.{c}: {raw} values → {norm} after normalisation")

print("Vérification terminée")

# 2.4 Verify types

In [0]:
for name, df in tables.items():
    print(f"\n=== {name} ===")
    display(df.limit(1))

- contracts : Contract_Start_Date	Contract_End_Date	Company_Start_Date	Birth_Date	are strings, they should be dates
- work_plan : Valid_From and Valid_To are string, should be dates, and Working_Days_per_Week should be an int 
- salary: Period is a string → convert to date (check format: MM/dd/yyyy vs dd/MM/yyyy). Gross_Salary, Net_Salary, Gross_Salary_108 are strings with comma decimals → replace , by . then cast to double.
- absence_types, postcodes: all columns stay string (codes and labels).

- abscences
![image_1791049445497.png](./image_1791049445497.png "image_1791049445497.png")

In [0]:
from pyspark.sql.functions import col, split, max as _max

def check_date_format(df, c):
    parts = df.select(
        split(col(c), "/").getItem(0).cast("int").alias("A"),
        split(col(c), "/").getItem(1).cast("int").alias("B"),
    )
    row = parts.agg(_max("A").alias("max_A"), _max("B").alias("max_B")).collect()[0]

    if row["max_A"] > 12:
        verdict = "dd/MM/yyyy"
    elif row["max_B"] > 12:
        verdict = "MM/dd/yyyy"
    else:
        verdict = "⚠️ ambigu"
    print(f"{c:<25} max A = {row['max_A']:>3}   max B = {row['max_B']:>3}   → {verdict}")

In [0]:
date_columns = {
    "contracts": (df_contracts, ["Contract_Start_Date", "Contract_End_Date",
                                 "Company_Start_Date", "Birth_Date"]),
    "workplan":  (df_workplan,  ["Valid_From", "Valid_To"]),
    "salary":    (df_salary,    ["Period"]),
    "absences":  (df_absences,  ["Date", "Period"]),
}

for name, (df, cols) in date_columns.items():
    print(f"\n=== {name} ===")
    for c in cols:
        check_date_format(df, c)

let's compare B with "Month"

In [0]:
from pyspark.sql.functions import expr

mismatch = df_absences.filter(
    expr("try_cast(split(Date, '/')[1] AS INT) != try_cast(Month AS INT)")
).count()

print(f"B ≠ Month : {mismatch}")   # 0 → B est bien le mois

All are in format dd/MM/yyyy

## 2.4.1. Contracts

In [0]:
from pyspark.sql.functions import col, expr

date_cols = ["Contract_Start_Date", "Contract_End_Date", "Company_Start_Date", "Birth_Date"]

df_contracts_types_corrected = df_contracts
for c in date_cols:
    df_contracts_types_corrected = df_contracts_types_corrected.withColumn(
        c, expr(f"try_to_date({c}, 'dd/MM/yyyy')")
    )
display(df_contracts_types_corrected.limit(1))

## 2.4.2. Work_plan

In [0]:
from pyspark.sql.functions import col, expr, regexp_replace, trim

date_cols = ["Valid_From", "Valid_To"]

df_workplan_types_corrected = df_workplan
for c in date_cols:
    df_workplan_types_corrected = df_workplan_types_corrected.withColumn(
        c, expr(f"try_to_date({c}, 'dd/MM/yyyy')")
    )

# Working_Days_per_Week : virgule → point, puis double
df_workplan_types_corrected = df_workplan_types_corrected.withColumn(
    "Working_Days_per_Week",
    expr("try_cast(regexp_replace(Working_Days_per_Week, ',', '.') AS DOUBLE)")
)

display(df_workplan_types_corrected.limit(1))

### 2.4.3. Salary

In [0]:
from pyspark.sql.functions import col, expr, trim

amount_cols = ["Gross_Salary", "Net_Salary", "Gross_Salary_108"]

# Period → date
df_salary_types_corrected = df_salary.withColumn(
    "Period", expr("try_to_date(Period, 'dd/MM/yyyy')")
)

# Montants : virgule → point, puis DECIMAL
for c in amount_cols:
    df_salary_types_corrected = df_salary_types_corrected.withColumn(
        c, expr(f"try_cast(regexp_replace({c}, ',', '.') AS DECIMAL(12,2))")
    )
display(df_salary_types_corrected.limit(1))

## 2.4.4. Abscences

In [0]:
from pyspark.sql.functions import col, expr, trim

# 0. Corriger la faute de frappe
df_abs = df_absences.withColumnRenamed("Freq_Z3_Daqs", "Freq_Z3_Days")

# 1. Repérer les colonnes par type
date_cols   = ["Date", "Period"]
int_cols    = ["Year", "Quarter", "Month"] + [c for c in df_abs.columns if c.startswith("Freq_")]
double_cols = [c for c in df_abs.columns if c.startswith("Qty_")]   # inclut Qty_Days_Worked, Qty_Working_Days

# 2. Conversion
df_absences_types_corrected = df_abs

for c in date_cols:
    df_absences_types_corrected = df_absences_types_corrected.withColumn(
        c, expr(f"try_to_date({c}, 'dd/MM/yyyy')")
    )

for c in int_cols:
    df_absences_types_corrected = df_absences_types_corrected.withColumn(
        c, expr(f"try_cast({c} AS INT)")
    )

for c in double_cols:
    df_absences_types_corrected = df_absences_types_corrected.withColumn(
        c, expr(f"try_cast(regexp_replace({c}, ',', '.') AS DOUBLE)")
    )

print("Dates   :", date_cols)
print("Int     :", int_cols)
print("Double  :", double_cols)
display(df_absences_types_corrected.limit(1))

In [0]:
# No change all in string
df_postcodes_types_corrected     = df_postcodes
df_abs_types_types_corrected = df_abs_types

In [0]:
tables_typed = {
    "contracts":     df_contracts_types_corrected,
    "workplan":      df_workplan_types_corrected,
    "salary":        df_salary_types_corrected,
    "absences":      df_absences_types_corrected,
    "absence_types": df_abs_types_types_corrected,
    "postcodes":     df_postcodes_types_corrected,
}


## 2.5. Verify missing values

In [0]:
from pyspark.sql.functions import col, sum as _sum, trim, when

def verify_na(dfs):
    for name, df in dfs.items():
        cols = [c for c in df.columns if not c.startswith("_")]
        total = df.count()

        na_counts = df.select([
            _sum(when(col(c).isNull() | (trim(col(c)) == ""), 1).otherwise(0)).alias(c)
            for c in cols
        ]).collect()[0]  # 1 if NA else 0, then sum

        print(f"\n=== {name} ({total} rows) ===")
        found = False
        for c in cols:
            n = na_counts[c]
            if n > 0:
                print(f"  {c:<30} {n:>8}  ({n / total:.1%})")
                found = True
        if not found:
            print("  No nulls found")

In [0]:
verify_na(dfs=tables_typed)

contracts and work_plan